In [1]:
import os
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.models import resnet34, ResNet34_Weights


@dataclass
class CFG:
    data_dir: str = "./data"
    num_classes: int = 10
    image_size: int = 224          # ResNet ImageNet default; we upscale CIFAR-10
    batch_size: int = 128
    num_workers: int = 4
    epochs: int = 30

    lr_head: float = 1e-3          # higher LR for classifier head
    lr_backbone: float = 1e-4      # lower LR for pretrained backbone
    weight_decay: float = 5e-4

    warmup_head_epochs: int = 2    # first N epochs: train only fc head (optional)
    label_smoothing: float = 0.0

    use_amp: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def build_dataloaders(cfg: CFG):
    # ImageNet normalization (since backbone pretrained on ImageNet)
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    train_tfms = T.Compose([
        T.Resize(cfg.image_size),
        T.RandomCrop(cfg.image_size, padding=16),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(imagenet_mean, imagenet_std),
    ])

    test_tfms = T.Compose([
        T.Resize(cfg.image_size),
        T.CenterCrop(cfg.image_size),
        T.ToTensor(),
        T.Normalize(imagenet_mean, imagenet_std),
    ])

    train_set = torchvision.datasets.CIFAR10(
        root=cfg.data_dir, train=True, download=True, transform=train_tfms
    )
    test_set = torchvision.datasets.CIFAR10(
        root=cfg.data_dir, train=False, download=True, transform=test_tfms
    )

    train_loader = DataLoader(
        train_set, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_set, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True
    )
    return train_loader, test_loader


def build_model(cfg: CFG):
    weights = ResNet34_Weights.IMAGENET1K_V1
    model = resnet34(weights=weights)

    # Replace classifier for CIFAR-10
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, cfg.num_classes)
    return model


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return loss_sum / total, correct / total


def set_backbone_trainable(model: nn.Module, trainable: bool):
    # Freeze/unfreeze everything except classifier head
    for name, p in model.named_parameters():
        if name.startswith("fc."):
            p.requires_grad = True
        else:
            p.requires_grad = trainable


def make_optimizer(cfg: CFG, model: nn.Module):
    # Two parameter groups: backbone (lower LR), head (higher LR)
    backbone_params = []
    head_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("fc."):
            head_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": cfg.lr_backbone})
    if head_params:
        param_groups.append({"params": head_params, "lr": cfg.lr_head})

    optimizer = optim.AdamW(param_groups, weight_decay=cfg.weight_decay)
    return optimizer


def train_one_epoch(model, loader, optimizer, scaler, device, cfg: CFG):
    model.train()
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(cfg.use_amp and device.startswith("cuda"))):
            logits = model(x)
            loss = criterion(logits, y)

        if cfg.use_amp and device.startswith("cuda"):
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    return loss_sum / total, correct / total


def main():
    cfg = CFG()
    set_seed(cfg.seed)

    train_loader, test_loader = build_dataloaders(cfg)
    model = build_model(cfg).to(cfg.device)

    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and cfg.device.startswith("cuda")))

    best_acc = 0.0
    for epoch in range(cfg.epochs):
        # Optional: warmup by training only the classifier head for first N epochs
        if epoch < cfg.warmup_head_epochs:
            set_backbone_trainable(model, trainable=False)
        else:
            set_backbone_trainable(model, trainable=True)

        optimizer = make_optimizer(cfg, model)

        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler, cfg.device, cfg)
        te_loss, te_acc = evaluate(model, test_loader, cfg.device)
        dt = time.time() - t0

        if te_acc > best_acc:
            best_acc = te_acc
            os.makedirs("checkpoints", exist_ok=True)
            torch.save({"model": model.state_dict(), "cfg": cfg.__dict__},
                       "checkpoints/resnet34_cifar10_best.pt")

        print(f"Epoch {epoch+1:02d}/{cfg.epochs} | "
              f"train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | "
              f"test loss {te_loss:.4f} acc {te_acc*100:.2f}% | "
              f"best {best_acc*100:.2f}% | {dt:.1f}s")

    print("Done. Best test acc:", best_acc)


if __name__ == "__main__":
    main()


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /home/sgchr/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:07<00:00, 11.7MB/s]
/tmp/ipykernel_2734484/987800778.py:174: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and cfg.device.startswith("cuda")))


Epoch 01/30 | train loss 0.9061 acc 72.06% | test loss 0.6383 acc 78.95% | best 78.95% | 34.7s
Epoch 02/30 | train loss 0.6114 acc 79.62% | test loss 0.6016 acc 79.54% | best 79.54% | 32.7s
Epoch 03/30 | train loss 0.2227 acc 92.38% | test loss 0.1658 acc 94.44% | best 94.44% | 82.2s
Epoch 04/30 | train loss 0.1327 acc 95.52% | test loss 0.1570 acc 94.78% | best 94.78% | 70.5s
Epoch 05/30 | train loss 0.0971 acc 96.68% | test loss 0.1702 acc 94.85% | best 94.85% | 70.6s
Epoch 06/30 | train loss 0.0806 acc 97.22% | test loss 0.1649 acc 94.93% | best 94.93% | 70.7s
Epoch 07/30 | train loss 0.0665 acc 97.79% | test loss 0.1543 acc 95.12% | best 95.12% | 70.9s
Epoch 08/30 | train loss 0.0560 acc 98.09% | test loss 0.1729 acc 95.10% | best 95.12% | 70.9s
Epoch 09/30 | train loss 0.0512 acc 98.30% | test loss 0.1575 acc 95.59% | best 95.59% | 70.6s
Epoch 10/30 | train loss 0.0449 acc 98.48% | test loss 0.1566 acc 95.37% | best 95.59% | 70.7s
Epoch 11/30 | train loss 0.0433 acc 98.47% | test 

In [2]:
import os
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, resnet34, ResNet18_Weights, ResNet34_Weights


@dataclass
class TrainConfig:
    data_dir: str = "./data"
    num_classes: int = 100

    image_size: int = 224
    batch_size: int = 128
    num_workers: int = 2
    epochs: int = 40

    lr_head: float = 1e-3
    lr_backbone: float = 1e-4
    weight_decay: float = 5e-4

    warmup_head_epochs: int = 2
    label_smoothing: float = 0.0

    use_amp: bool = True
    seed: int = 42
    save_dir: str = "./checkpoints"


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def build_cifar100_loaders(cfg: TrainConfig):
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    train_tfms = T.Compose([
        T.Resize(cfg.image_size),
        T.RandomCrop(cfg.image_size, padding=16),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(imagenet_mean, imagenet_std),
    ])

    test_tfms = T.Compose([
        T.Resize(cfg.image_size),
        T.CenterCrop(cfg.image_size),
        T.ToTensor(),
        T.Normalize(imagenet_mean, imagenet_std),
    ])

    train_set = torchvision.datasets.CIFAR100(
        root=cfg.data_dir, train=True, download=True, transform=train_tfms
    )
    test_set = torchvision.datasets.CIFAR100(
        root=cfg.data_dir, train=False, download=True, transform=test_tfms
    )

    train_loader = DataLoader(
        train_set,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True
    )
    return train_loader, test_loader


def build_resnet(arch: str, num_classes: int):
    arch = arch.lower().strip()
    if arch in ["resnet18", "18", "r18"]:
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        arch_name = "resnet18"
    elif arch in ["resnet34", "34", "r34"]:
        model = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        arch_name = "resnet34"
    else:
        raise ValueError("arch must be one of: 'resnet18' or 'resnet34' (also accepts '18','34','r18','r34').")

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model, arch_name


def set_backbone_trainable(model: nn.Module, trainable: bool):
    for name, p in model.named_parameters():
        if name.startswith("fc."):
            p.requires_grad = True
        else:
            p.requires_grad = trainable


def make_optimizer(cfg: TrainConfig, model: nn.Module):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("fc."):
            head_params.append(p)
        else:
            backbone_params.append(p)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": cfg.lr_backbone})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.lr_head})

    return optim.AdamW(groups, weight_decay=cfg.weight_decay)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, loss_sum = 0, 0, 0.0

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    return loss_sum / total, correct / total


def train_one_epoch(model, loader, optimizer, scaler, device, cfg: TrainConfig):
    model.train()
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

    correct, total, loss_sum = 0, 0, 0.0
    use_amp = cfg.use_amp and device.type == "cuda"

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    return loss_sum / total, correct / total


def train_cifar100_resnet(arch: str = "resnet18", cfg: TrainConfig = None):
    """
    Notebook-friendly training entry point.

    Example:
        cfg = TrainConfig(epochs=10, batch_size=64)
        train_cifar100_resnet("resnet34", cfg)
    """
    if cfg is None:
        cfg = TrainConfig()

    set_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, test_loader = build_cifar100_loaders(cfg)
    model, arch_name = build_resnet(arch, cfg.num_classes)
    model = model.to(device)

    os.makedirs(cfg.save_dir, exist_ok=True)
    best_acc = 0.0
    best_path = os.path.join(cfg.save_dir, f"{arch_name}_cifar100_best.pt")

    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))

    for epoch in range(cfg.epochs):
        # Warmup: train only head first
        if epoch < cfg.warmup_head_epochs:
            set_backbone_trainable(model, trainable=False)
        else:
            set_backbone_trainable(model, trainable=True)

        optimizer = make_optimizer(cfg, model)

        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler, device, cfg)
        te_loss, te_acc = evaluate(model, test_loader, device)
        dt = time.time() - t0

        if te_acc > best_acc:
            best_acc = te_acc
            torch.save(
                {"model": model.state_dict(), "arch": arch_name, "cfg": cfg.__dict__},
                best_path
            )

        print(
            f"[{arch_name}] Epoch {epoch+1:02d}/{cfg.epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | "
            f"test loss {te_loss:.4f} acc {te_acc*100:.2f}% | "
            f"best {best_acc*100:.2f}% | {dt:.1f}s"
        )

    print(f"Done. Best test acc: {best_acc*100:.2f}%")
    print(f"Best checkpoint saved to: {best_path}")
    return model, best_path, best_acc


In [3]:
cfg = TrainConfig(epochs=10, batch_size=128, image_size=224)
model18, ckpt18, acc18 = train_cifar100_resnet("resnet18", cfg)

model34, ckpt34, acc34 = train_cifar100_resnet("resnet34", cfg)


100%|██████████| 169M/169M [00:19<00:00, 8.66MB/s] 
/tmp/ipykernel_2734484/1141359986.py:201: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


[resnet18] Epoch 01/10 | train loss 2.6511 acc 39.45% | test loss 1.9073 acc 50.82% | best 50.82% | 46.0s
[resnet18] Epoch 02/10 | train loss 1.7444 acc 54.13% | test loss 1.6572 acc 55.11% | best 55.11% | 46.3s
[resnet18] Epoch 03/10 | train loss 0.9589 acc 71.24% | test loss 0.8026 acc 75.45% | best 75.45% | 47.7s
[resnet18] Epoch 04/10 | train loss 0.6305 acc 80.27% | test loss 0.7813 acc 76.98% | best 76.98% | 47.9s
[resnet18] Epoch 05/10 | train loss 0.4647 acc 85.11% | test loss 0.7548 acc 77.94% | best 77.94% | 48.0s
[resnet18] Epoch 06/10 | train loss 0.3480 acc 88.70% | test loss 0.7437 acc 79.34% | best 79.34% | 47.6s
[resnet18] Epoch 07/10 | train loss 0.2686 acc 91.22% | test loss 0.7642 acc 79.22% | best 79.34% | 47.6s
[resnet18] Epoch 08/10 | train loss 0.2150 acc 93.02% | test loss 0.7954 acc 79.52% | best 79.52% | 47.6s
[resnet18] Epoch 09/10 | train loss 0.1731 acc 94.20% | test loss 0.8439 acc 79.67% | best 79.67% | 47.7s
[resnet18] Epoch 10/10 | train loss 0.1462 acc


training vitb on cifar10 from scratch

In [1]:
# finetune_vit_b16_cifar10.py
# Finetune torchvision ViT B 16 pretrained on ImageNet 1K for CIFAR 10

import os
import time
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.models import vit_b_16, ViT_B_16_Weights


def accuracy_top1(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(images)
        loss = loss_fn(logits, targets)

        total_loss += loss.item()
        total_acc += accuracy_top1(logits, targets)
        n_batches += 1

    return total_loss / max(1, n_batches), total_acc / max(1, n_batches)


def main():
    # -------- config you may want to change --------
    data_dir = "./data"
    batch_size = 128
    num_workers = 4
    epochs = 20
    lr = 5e-4
    weight_decay = 0.05
    label_smoothing = 0.1
    freeze_backbone = False  # set True for linear probing first
    amp = True  # mixed precision
    # ---------------------------------------------

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.backends.cudnn.benchmark = True

    # ImageNet normalization since we use ImageNet pretrained weights
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    test_tfms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    train_set = CIFAR10(root=data_dir, train=True, download=True, transform=train_tfms)
    test_set = CIFAR10(root=data_dir, train=False, download=True, transform=test_tfms)

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True
    )

    # Load pretrained ViT B 16
    weights = ViT_B_16_Weights.IMAGENET1K_V1
    model = vit_b_16(weights=weights)

    # Replace classifier head for CIFAR 10
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, 10)

    if freeze_backbone:
        for name, p in model.named_parameters():
            p.requires_grad = False
        for p in model.heads.head.parameters():
            p.requires_grad = True

    model.to(device)

    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # AdamW is standard for ViT finetuning
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    # Cosine LR schedule
    steps_per_epoch = len(train_loader)
    total_steps = epochs * steps_per_epoch
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    scaler = torch.cuda.amp.GradScaler(enabled=(amp and device == "cuda"))

    best_acc = 0.0
    os.makedirs("checkpoints", exist_ok=True)

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()

        running_loss = 0.0
        running_acc = 0.0

        for step, (images, targets) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(amp and device == "cuda")):
                logits = model(images)
                loss = loss_fn(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            scheduler.step()

            running_loss += loss.item()
            running_acc += accuracy_top1(logits, targets)

            if step % 100 == 0 or step == steps_per_epoch:
                avg_loss = running_loss / step
                avg_acc = running_acc / step
                current_lr = optimizer.param_groups[0]["lr"]
                print(
                    f"epoch {epoch:02d} step {step:04d}/{steps_per_epoch} "
                    f"lr {current_lr:.2e} loss {avg_loss:.4f} acc {avg_acc:.4f}"
                )

        val_loss, val_acc = evaluate(model, test_loader, device)
        dt = time.time() - t0
        print(f"epoch {epoch:02d} done in {dt:.1f}s | val loss {val_loss:.4f} | val acc {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            ckpt_path = os.path.join("checkpoints", "vit_b16_cifar10_best.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "best_acc": best_acc,
                },
                ckpt_path,
            )
            print(f"saved best checkpoint to {ckpt_path} (acc {best_acc:.4f})")

    print(f"best val acc: {best_acc:.4f}")


if __name__ == "__main__":
    main()


/tmp/ipykernel_2946276/3571102458.py:118: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp and device == "cuda"))
/tmp/ipykernel_2946276/3571102458.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(amp and device == "cuda")):


epoch 01 step 0100/391 lr 5.00e-04 loss 2.1289 acc 0.2300
epoch 01 step 0200/391 lr 4.99e-04 loss 2.0731 acc 0.2486
epoch 01 step 0300/391 lr 4.98e-04 loss 2.0021 acc 0.2835
epoch 01 step 0391/391 lr 4.97e-04 loss 1.9456 acc 0.3129
epoch 01 done in 260.1s | val loss 1.5871 | val acc 0.4299
saved best checkpoint to checkpoints/vit_b16_cifar10_best.pt (acc 0.4299)
epoch 02 step 0100/391 lr 4.95e-04 loss 1.6768 acc 0.4518
epoch 02 step 0200/391 lr 4.93e-04 loss 1.6407 acc 0.4714
epoch 02 step 0300/391 lr 4.90e-04 loss 1.6195 acc 0.4798
epoch 02 step 0391/391 lr 4.88e-04 loss 1.6005 acc 0.4883
epoch 02 done in 260.1s | val loss 1.2994 | val acc 0.5392
saved best checkpoint to checkpoints/vit_b16_cifar10_best.pt (acc 0.5392)
epoch 03 step 0100/391 lr 4.84e-04 loss 1.5018 acc 0.5402
epoch 03 step 0200/391 lr 4.81e-04 loss 1.4885 acc 0.5476
epoch 03 step 0300/391 lr 4.77e-04 loss 1.4693 acc 0.5568
epoch 03 step 0391/391 lr 4.73e-04 loss 1.4576 acc 0.5623
epoch 03 done in 259.4s | val loss 1.1

Finetuning of vitb16 on cifar10

In [1]:
# finetune_vit_b16_cifar10_method1_nodropout.py
# Method 1 style: head warmup (frozen backbone) -> full finetune
# NO dropout. ViT-B/16 pretrained on ImageNet1K, finetuned on CIFAR-10.

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.backends.cudnn.benchmark = True

    # ----------------------------
    # 1) Load ImageNet-pretrained ViT-B/16
    # ----------------------------
    weights = ViT_B_16_Weights.DEFAULT
    model = vit_b_16(weights=weights)

    # ----------------------------
    # 2) Replace classifier head for CIFAR-10 (NO dropout)
    # ----------------------------
    num_classes = 10
    in_features = model.heads.head.in_features  # typically 768
    model.heads = nn.Sequential(
        nn.Linear(in_features, num_classes),
    )
    model = model.to(device)

    # ----------------------------
    # 3) CIFAR-10 transforms (224x224 + ImageNet normalization)
    # ----------------------------
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)

    train_tfm = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    # Use torchvision's official preset for this weight
    test_tfm = weights.transforms()

    train_set = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tfm)
    test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tfm)

    BATCH_SIZE = 128
    NUM_WORKERS = 4

    train_loader = DataLoader(
        train_set, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    test_loader = DataLoader(
        test_set, batch_size=128, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )

    # ----------------------------
    # Helpers
    # ----------------------------
    def set_backbone_trainable(m: nn.Module, trainable: bool):
        for name, p in m.named_parameters():
            if name.startswith("heads."):  # always train head
                p.requires_grad = True
            else:
                p.requires_grad = trainable

    @torch.no_grad()
    def evaluate() -> float:
        model.eval()
        correct = 0
        total = 0
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
        return correct / max(1, total)

    criterion = nn.CrossEntropyLoss()

    use_amp = (device == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    os.makedirs("checkpoints", exist_ok=True)

    # ----------------------------
    # 4) Warmup: head-only (2 epochs)
    # ----------------------------
    set_backbone_trainable(model, trainable=False)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
        weight_decay=0.05
    )

    for epoch in range(2):
        model.train()
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(x), y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        acc = evaluate()
        print(f"Warmup epoch {epoch+1}: acc={acc:.4f}")

    # ----------------------------
    # 5) Full fine-tune: unfreeze (10 epochs)
    # ----------------------------
    set_backbone_trainable(model, trainable=True)

    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

    best_acc = 0.0
    best_path = os.path.join("checkpoints", "vit_b16_cifar10_best_finetuning.pt")

    for epoch in range(10):
        model.train()
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(x), y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        scheduler.step()

        acc = evaluate()
        print(f"Finetune epoch {epoch+1}: acc={acc:.4f}")

        # Save best checkpoint (state_dict only)
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), best_path)
            print(f"Saved best to {best_path} (acc {best_acc:.4f})")

    # Final save
    final_path = os.path.join("checkpoints", "vit_b16_cifar10_final_nodropout.pt")
    torch.save(model.state_dict(), final_path)
    print(f"Saved final checkpoint to {final_path}")
    print(f"Best val acc: {best_acc:.4f}")

if __name__ == "__main__":
    main()


/tmp/ipykernel_3014285/4260757076.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_3014285/4260757076.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Warmup epoch 1: acc=0.9411
Warmup epoch 2: acc=0.9454


/tmp/ipykernel_3014285/4260757076.py:142: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Finetune epoch 1: acc=0.9691
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9691)
Finetune epoch 2: acc=0.9765
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9765)
Finetune epoch 3: acc=0.9717
Finetune epoch 4: acc=0.9753
Finetune epoch 5: acc=0.9788
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9788)
Finetune epoch 6: acc=0.9796
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9796)
Finetune epoch 7: acc=0.9798
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9798)
Finetune epoch 8: acc=0.9825
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9825)
Finetune epoch 9: acc=0.9841
Saved best to checkpoints/vit_b16_cifar10_best_finetuning.pt (acc 0.9841)
Finetune epoch 10: acc=0.9833
Saved final checkpoint to checkpoints/vit_b16_cifar10_final_nodropout.pt
Best val acc: 0.9841
